# Kaggle Launcher — E2VID Reconstruction + YOLO Training
**Last updated: 2026-06-16  v35**

This notebook is a thin wrapper that runs the two Python scripts on Kaggle GPU.

**Before running:**
1. Confirm these three datasets are attached as inputs:
   - `gennepy/fred-events-ami` — events + coordinates
   - `gennepy/fred-scripts-ami` — reconstruct.py, train_yolo.py
   - `gennepy/fred-frames-all` — all 5 sequences pre-reconstructed (skips reconstruction)
2. Set Runtime → Accelerator → **GPU T4 x2** (or P100)
3. Run all cells top to bottom

**What this run does (Run 5a):**
- Skips reconstruction (frames restored from `fred-frames-all`)
- Trains YOLOv8s on sequences 84, 85, 124, 201 (val: 127) — same data as Run 4, larger model
- optimizer=SGD, lr0=0.01, mosaic=1.0, mixup=0.0
- Saves weights, KPIs, curves, and val previews; deletes frames to fit 20 GB output cap

**After completion:**
```bash
bash scripts/sync_from_kaggle.sh
docker compose build e2vid && docker compose up -d
```

## 1 · Configuration — edit this cell

In [ ]:
from pathlib import Path
import datetime

# ── Kaggle dataset paths ───────────────────────────────────────────────────────
AMI_INPUT        = Path('/kaggle/input/datasets/gennepy/fred-events-ami')
SCRIPTS_INPUT    = Path('/kaggle/input/datasets/gennepy/fred-scripts-ami')
AMI_WORK         = Path('/kaggle/working')
PREV_RECON_INPUT = Path('/kaggle/input/datasets/gennepy/fred-frames-all')

# ── Run control ───────────────────────────────────────────────────────────────
RESUME        = False
SKIP_TRAINING = False

# ── Sequences ─────────────────────────────────────────────────────────────────
# Run 5a: same data as Run 4, upgraded model (YOLOv8n → YOLOv8s)
# Run 5b: new sequences — change these after reconstruction of 44/45/46/47/146
SEQUENCES     = ['sequence_84', 'sequence_85', 'sequence_124', 'sequence_201', 'sequence_127']
VAL_SEQUENCES = ['sequence_127']

# ── Training parameters ───────────────────────────────────────────────────────
MODEL  = 'yolov8s.pt'   # Run 4 used yolov8n.pt
EPOCHS = 100
BATCH  = 16

# ── Reconstruction parameters ─────────────────────────────────────────────────
START_S          = 5.0
EVENTS_PER_PIXEL = 0.01
SMOKE_EVENTS     = None   # e.g. 100_000 for a quick smoke test (~10 frames)

# ── Derived paths ─────────────────────────────────────────────────────────────
SCRIPTS_DIR = SCRIPTS_INPUT
EVENTS_ROOT = AMI_INPUT / 'data' / 'processed'
RAW_ROOT    = AMI_INPUT / 'data' / 'raw'
RECON_ROOT  = AMI_WORK  / 'data' / 'processed'
WEIGHTS_OUT = AMI_WORK  / 'yolo_e2vid.pt'
RUNS_DIR    = AMI_WORK  / 'yolo_runs'
LOG_FILE    = AMI_WORK  / 'logs' / f'run_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
DATASET_DIR = AMI_WORK  / 'yolo_e2vid'
WORK_DIR    = AMI_WORK  / 'work'

WORK_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

train_seqs = [s for s in SEQUENCES if s not in VAL_SEQUENCES]
print('Model         :', MODEL)
print('Train seqs    :', train_seqs)
print('Val seqs      :', VAL_SEQUENCES)
print('Epochs        :', EPOCHS, '  Batch:', BATCH)
print('Resume        :', RESUME)
print('Skip training :', SKIP_TRAINING)
print('Smoke events  :', SMOKE_EVENTS or 'full run')

## 2 · Install dependencies

In [ ]:
!pip install -q h5py ultralytics==8.4.54 imageio scikit-image pandas matplotlib
import torch
print(f'PyTorch {torch.__version__} — CUDA: {torch.cuda.is_available()}')

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit(
        'No GPU detected. Go to Settings → Accelerator → GPU T4 x2, then restart.'
    )
print(f'GPU: {torch.cuda.get_device_name(0)}  '
      f'({torch.cuda.get_device_properties(0).total_memory // 1024**2} MB)')

## 3 · Helpers

In [ ]:
import os, subprocess, sys, datetime, shutil

# Copy scripts from input dataset to local working dir
LOCAL_SCRIPTS = Path('/kaggle/working/scripts')
LOCAL_SCRIPTS.mkdir(exist_ok=True)
for script in ['reconstruct.py', 'train_yolo.py']:
    shutil.copy(SCRIPTS_DIR / script, LOCAL_SCRIPTS / script)
print(f'Scripts copied to {LOCAL_SCRIPTS}')

def run_streaming(cmd):
    """Run a command, stream output to notebook and append to log file."""
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    with open(LOG_FILE, 'a') as lf:
        for line in process.stdout:
            print(line, end='', flush=True)
            lf.write(line)
            lf.flush()
    process.wait()
    return process.returncode

def log(msg):
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    line = f'[{ts}] {msg}'
    print(line)
    with open(LOG_FILE, 'a') as f:
        f.write(line + '\n')

log('Helpers ready.')

## 4 · Cleanup — remove stale output before each run

In [ ]:
import shutil

if RESUME:
    print('RESUME=True — skipping cleanup, keeping existing outputs.')
else:
    for seq in SEQUENCES:
        recon_dir = RECON_ROOT / seq / 'reconstruction_e2vid'
        if recon_dir.exists():
            shutil.rmtree(recon_dir)
            print(f'Cleaned: {recon_dir}')
        else:
            print(f'Nothing to clean: {recon_dir}')

    shutil.rmtree(DATASET_DIR, ignore_errors=True)
    print(f'Cleaned: {DATASET_DIR}')

    shutil.rmtree(WORK_DIR / 'rpg_e2vid', ignore_errors=True)
    print(f'Cleaned: {WORK_DIR / "rpg_e2vid"}')

In [ ]:
# ── Restore previous reconstructions so they are not re-run ─────────────────
import shutil

if PREV_RECON_INPUT is not None and PREV_RECON_INPUT.exists():
    log('Restoring previous reconstructions from ' + str(PREV_RECON_INPUT))
    log('  Contents: ' + str([p.name for p in sorted(PREV_RECON_INPUT.iterdir())[:10]]))
    for seq in SEQUENCES:
        # Try all known layouts (--dir-mode zip strips the leading data/ prefix)
        candidates = [
            PREV_RECON_INPUT / 'processed' / seq / 'reconstruction_e2vid',
            PREV_RECON_INPUT / 'data' / 'processed' / seq / 'reconstruction_e2vid',
            PREV_RECON_INPUT / seq,
        ]
        src_dir = next((p for p in candidates if p.exists()), None)
        dst_dir = RECON_ROOT / seq / 'reconstruction_e2vid'
        if src_dir is not None and not dst_dir.exists():
            dst_dir.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(src_dir, dst_dir)
            n = len(list(dst_dir.glob('frame_*')))
            log(f'  {seq}: restored {n} frames')
        elif dst_dir.exists():
            log(f'  {seq}: already in working dir, skipping copy')
        else:
            log(f'  {seq}: not found in prev recon input — will reconstruct')
else:
    log('No PREV_RECON_INPUT — all sequences will be reconstructed from scratch')

## 5 · Reconstruct e2vid frames

In [ ]:
log('=== Reconstruction started ===')

for seq in SEQUENCES:
    zip_path = EVENTS_ROOT / seq / 'events.zip'    # read from input dataset
    out_dir  = RECON_ROOT  / seq / 'reconstruction_e2vid'  # write to working

    if out_dir.exists() and (any(out_dir.glob('frame_*.png')) or any(out_dir.glob('frame_*.jpg'))):
        log(f'{seq}: frames already exist — skipping reconstruction')
        continue

    log(f'=== Reconstructing {seq} ===')
    cmd = [
        sys.executable, str(LOCAL_SCRIPTS / 'reconstruct.py'),
        '--zip_path',         str(zip_path),
        '--out_dir',          str(out_dir),
        '--work_dir',         str(WORK_DIR),
        '--events_per_pixel', str(EVENTS_PER_PIXEL),
        '--compress_jpeg',    # convert PNGs → JPEG after each sequence (~5x smaller)
    ]
    if SMOKE_EVENTS:
        cmd += ['--max_events', str(SMOKE_EVENTS)]

    rc = run_streaming(cmd)
    if rc != 0:
        log(f'ERROR: reconstruct.py failed for {seq} (exit code {rc})')
        raise RuntimeError(f'reconstruct.py failed for {seq} (exit code {rc})')

log('=== Reconstruction done ===')

## 6 · Train YOLO

In [ ]:
if SKIP_TRAINING:
    log('SKIP_TRAINING=True — skipping training')
else:
    log('=== Training started ===')

    resume_yaml = DATASET_DIR / 'dataset.yaml'
    resume_yaml.parent.mkdir(parents=True, exist_ok=True)
    resume_yaml.write_text(f"""path: {DATASET_DIR}
train: train.txt
val:   val.txt

nc: 1
names:
  0: drone
""")
    log(f'dataset.yaml pre-written → {DATASET_DIR}')

    cmd = [
        sys.executable, str(LOCAL_SCRIPTS / 'train_yolo.py'),
        '--sequences',  *SEQUENCES,
        '--val_sequences', *VAL_SEQUENCES,
        '--raw_root',   str(RAW_ROOT),
        '--recon_root', str(RECON_ROOT),
        '--out_dir',    str(DATASET_DIR),
        '--runs_dir',   str(RUNS_DIR),
        '--weights',    str(WEIGHTS_OUT),
        '--model',      MODEL,
        '--epochs',     str(EPOCHS),
        '--batch',      str(BATCH),
    ]

    if RESUME:
        cmd += ['--resume']

    rc = run_streaming(cmd)
    if rc != 0:
        log(f'ERROR: train_yolo.py failed (exit code {rc})')
        raise RuntimeError('train_yolo.py failed')

    log(f'=== Training done — weights at {WEIGHTS_OUT} ===')

In [ ]:
# Free disk space so yolo_e2vid.pt fits in the Kaggle output snapshot.
import shutil, os

freed = 0

def _rmdir(p):
    global freed
    p = Path(p)
    if p.exists():
        size = sum(f.stat().st_size for f in p.rglob('*') if f.is_file())
        shutil.rmtree(p)
        freed += size
        log(f'  deleted {p}  ({size / 1e9:.2f} GB)')

def _rm(p):
    global freed
    p = Path(p)
    if p.is_file():
        freed += p.stat().st_size
        p.unlink()

# ── Save YOLO outputs to kpis/ before cleanup ────────────────────────────────
e2vid_run = RUNS_DIR / 'e2vid'
kpis_dir  = AMI_WORK / 'kpis'
kpis_dir.mkdir(parents=True, exist_ok=True)

KEEP = {
    # metrics
    'results.csv', 'results.png',
    # confusion matrices
    'confusion_matrix.png', 'confusion_matrix_normalized.png',
    # PR / F1 / P / R curves
    'BoxF1_curve.png', 'BoxP_curve.png', 'BoxR_curve.png', 'BoxPR_curve.png',
    # label distribution
    'labels.jpg', 'labels_correlogram.jpg',
    # training batch samples
    'train_batch0.jpg', 'train_batch1.jpg', 'train_batch2.jpg',
    # val predictions vs ground truth
    'val_batch0_labels.jpg', 'val_batch0_pred.jpg',
    'val_batch1_labels.jpg', 'val_batch1_pred.jpg',
    'val_batch2_labels.jpg', 'val_batch2_pred.jpg',
}

if e2vid_run.exists():
    for p in e2vid_run.iterdir():
        if p.is_file() and p.name in KEEP:
            shutil.copy(p, kpis_dir / p.name)
            log(f'  saved {p.name} → kpis/')

# ── Delete large directories ──────────────────────────────────────────────────
# Reconstructed frames (~10.5 GB) — already in fred-frames-all
_rmdir(RECON_ROOT.parent)

# Ultralytics per-epoch val images and per-batch plots in yolo_runs (~1 GB)
if e2vid_run.exists():
    for p in list(e2vid_run.iterdir()):
        if p.name != 'weights':
            _rmdir(p) if p.is_dir() else _rm(p)

# rpg_e2vid work dir + E2VID.pth.tar (~2 GB)
_rmdir(WORK_DIR)

log(f'Freed {freed / 1e9:.2f} GB total')
os.system('df -h /kaggle/working')

## 7 · Zip frames for download

Bundles all reconstructed JPEG frames into a single zip — much faster to download
than 40k individual files via the Kaggle API.


In [ ]:
# Disabled: frames are deleted by the disk-cleanup cell before this runs.
# Re-enable when a fresh reconstruction is needed and frames must be re-uploaded.
if False:
    # ── Zip reconstructed frames for fast single-file download ───────────────────
    import zipfile, os
    from pathlib import Path
    
    WORKING = Path('/kaggle/working')
    zip_path = WORKING / 'frames_e2vid.zip'
    
    log('=== Zipping reconstructed frames ===')
    frame_count = 0
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
        for seq in SEQUENCES:
            recon_dir = WORKING / 'data' / 'processed' / seq / 'reconstruction_e2vid'
            ts_file = recon_dir / 'timestamps.txt'
            if ts_file.exists():
                zf.write(ts_file, ts_file.relative_to(WORKING))
            for frame in sorted(recon_dir.glob('frame_*.jpg')):
                zf.write(frame, frame.relative_to(WORKING))
                frame_count += 1
    
    size_mb = zip_path.stat().st_size / 1e6
    log(f'frames_e2vid.zip: {frame_count} frames, {size_mb:.0f} MB → {zip_path}')
    

## 8 · Download results

After the notebook finishes, Kaggle saves `/kaggle/working/` as the run output.

**Weights + KPIs + logs (fast — ~50 MB):**
```bash
bash scripts/sync_from_kaggle.sh
```

**Reconstructed frames (~2.5 GB, single zip):**
```bash
bash scripts/sync_from_kaggle.sh --frames-zip
```

Then rebuild the service:
```bash
docker compose build e2vid && docker compose up -d
```
